<a href="https://colab.research.google.com/github/dan-the-man7/lab_4_/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q openai python-dotenv pandas matplotlib

import os, json, time, re, random

API_KEY = None
try:                                   # --- Google Colab (Secrets panel) ---
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # --- Local (.env file) ---
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.environ.get("Groq_API_key")

assert API_KEY, (
    "No API key found. In Colab add a secret named GROQ_API_KEY (key icon, "
    "left sidebar) and switch on 'Notebook access'. Locally, create a .env file."
)
print("Key loaded:", API_KEY[:4] + "..." + API_KEY[-4:])   # never print the whole

# OpenAI-compatible client pointed at Groq
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready. Model:", MODEL)

Key loaded: gsk_...ZNuR
Client ready. Model: llama-3.3-70b-versatile


In [ ]:
TOKEN_LOG = []          # (label, prompt_tokens, completion_tokens, total_tokens)


def ask_llm(user_prompt,
            system_prompt="You are a helpful assistant.",
            temperature=0.7,
            max_tokens=500,
            label="",
            retries=6):
    """Send one single-turn chat request and return the assistant's text."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            u = response.usage
            TOKEN_LOG.append((label or "unlabelled",
                              u.prompt_tokens, u.completion_tokens, u.total_tokens))
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt + random.random()
                print(f"   [rate limited — sleeping {wait:.1f}s]")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"ask_llm failed after {retries} attempts")


raw = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You explain finance simply, for a non-expert."},
        {"role": "user",   "content": "In two sentences, what is microfinance?"},
    ],
    temperature=0.7,
    max_tokens=150,
)

print("Answer:\n", raw.choices[0].message.content.strip())
print("\nRole of the reply:", raw.choices[0].message.role)
print("Finish reason     :", raw.choices[0].finish_reason)
print("\nUsage:")
print("  prompt_tokens     =", raw.usage.prompt_tokens)
print("  completion_tokens =", raw.usage.completion_tokens)
print("  total_tokens      =", raw.usage.total_tokens)

print("\nQuestion passed through ask_llm():\n",
      ask_llm("Name one risk a microfinance lender faces.",
              label="smoke_test", max_tokens=60))

Answer:
 Microfinance refers to the practice of providing small loans, savings, and other financial services to individuals or groups who lack access to traditional banking services, often in developing countries or low-income communities. The goal of microfinance is to empower these individuals, often entrepreneurs or small business owners, by giving them the financial tools and resources they need to improve their economic well-being and escape poverty.

Role of the reply: assistant
Finish reason     : stop

Usage:
  prompt_tokens     = 55
  completion_tokens = 77
  total_tokens      = 132

Question passed through ask_llm():
 One risk a microfinance lender faces is default risk, which is the risk that borrowers will be unable to repay their loans. This can be particularly challenging in microfinance, as borrowers often have limited financial resources and may be more vulnerable to economic shocks or other unexpected events that can affect their ability to repay


Section 1.1 Student Reasoning

A system role sets constraints or permissions that influence and govern the entire conversation. The user role contains the specific request for the LLM. A token is a chunk of text produced from a request of a set of words and sentences. Billing is based on token count because, the compute cost scales with the number of tokens processed. For instance, a 500-word prompt and a 5000-word prompt will not have the same token usage amount. The 5000-word request would require more tokens to serve, therefore culminating to a higher compute cost. The first call had a token count of 55 for the prompt_tokens

In [58]:
QUESTION = "Suggest a name for a savings product for market traders in Accra."
N_RUNS = 5

runs = {}
for temp in (0.0, 1.2):
  print(f"Running {N_RUNS} calls at temperature={temp} ...")
  runs[temp] = [
      ask_llm(QUESTION, temperature=temp, max_tokens=40, label=f"temp_{temp}")
      for _ in range(N_RUNS)
  ]

for temp, answers in runs.items():
  print("\n")
  print(f"Temperature = {temp}")
  print("\n")
  for i, a in enumerate(answers, 1):
    print(f"[{i}] {a}")

print(f"\n Distinct answers at temperature 0.0 : {len(set(runs[0.0]))} / {N_RUNS}")
print(f"\n Distinct answers at temperature 1.2 : {len(set(runs[1.2]))} / {N_RUNS}")

Running 5 calls at temperature=0.0 ...
   [rate limited — sleeping 1.5s]
   [rate limited — sleeping 2.2s]
   [rate limited — sleeping 4.6s]
   [rate limited — sleeping 8.5s]
   [rate limited — sleeping 16.2s]
Running 5 calls at temperature=1.2 ...
   [rate limited — sleeping 1.0s]
   [rate limited — sleeping 2.3s]
   [rate limited — sleeping 4.5s]
   [rate limited — sleeping 8.5s]
   [rate limited — sleeping 16.1s]
   [rate limited — sleeping 1.8s]
   [rate limited — sleeping 2.4s]
   [rate limited — sleeping 4.0s]
   [rate limited — sleeping 8.7s]
   [rate limited — sleeping 16.1s]
   [rate limited — sleeping 1.2s]
   [rate limited — sleeping 2.4s]
   [rate limited — sleeping 4.9s]
   [rate limited — sleeping 8.4s]
   [rate limited — sleeping 16.4s]
   [rate limited — sleeping 1.5s]
   [rate limited — sleeping 2.5s]
   [rate limited — sleeping 4.1s]
   [rate limited — sleeping 8.8s]
   [rate limited — sleeping 16.3s]
   [rate limited — sleeping 2.0s]
   [rate limited — sleeping 2.2s]

Student Reflection 1.2

At temperature 0.0, there is only 1 distinct answer out of the 5 answers. The remaining answers repeat what they are saying. At temperature 1.2, all 5 answers are distinct, with 2 answers suggesting the "Makola Savings", but providing different explanations for the name choice. For this regime, Temperature 0 will be more appropriate because, extraction has a single correct answer, therefore any differences across runs would be classified as errors, rather than creativity.

In [59]:
## Section 2

LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.


In [60]:
## Version 1

SUMMARY_SYSTEM_V1 = "You are a helpful assistant."
SUMMARY_PROMPT_V1 = "Summarise this:\n\n{letter}"

## Version 2
SUMMARY_SYSTEM_V2 = """You are an assistant to a loan officer at a Ghanaian microfinance institution. You write short factual briefs that let the officer triage applications quickly.

Rules you must follow:
1. Use only facts stated in the letter. Never invent, infer, estimate, or improve on a number, a business detail or a personal circumstance.
2. If an important item (Loan amount, income, collateral/guarantor, repayment plan) is not stated, say explicitly that it is not stated.
3. Stay neutral. No sympathy, no encouragement, and no view on whether the loan should be granted.
4. Output 3-4 sentences of plain prose. No headings, no bullet points, no preamble such as "Here is a summary"."""

SUMMARY_PROMPT_V2 = """Summarize the loan application below in 3-4 sentences.

Cover, in this order:
(a) who the applicant is and what business they run,
(b) how much they are requesting and for what purpose,
(c) the financial position they claim (income/profit/savings) and the repayment they propose,
(d) what security is offered, and any material information that is missing.

LOAN APPLICATION

{letter}"""

def summarize(letter_text, version="v2"):
  if version == "v1":
    return ask_llm(SUMMARY_PROMPT_V1.format(letter=letter_text), SUMMARY_SYSTEM_V1, temperature=0.7, max_tokens=300, label="summary_v1")

  return ask_llm(SUMMARY_PROMPT_V2.format(letter=letter_text), SUMMARY_SYSTEM_V2, temperature=0.0, max_tokens=300, label="summary_v2")


## Side by side comparison of L001 and L006
for lid in ["L002", "L006"]:
  print(f"LETTER {lid}")
  print("\nORIGINAL LETTER")
  print(LETTERS [lid])
  print("\nV1 (naive prompt, temperature 0.7)")
  print(summarize (LETTERS[lid], "v1"))
  print("\nV2 (engineered prompt, temperature 0.0)")
  print(summarize (LETTERS[lid], "v2"))
  print()

## final version used by the rest of the lab

SUMMARY_SYSTEM = SUMMARY_SYSTEM_V2
SUMMARY_PROMPT = SUMMARY_PROMPT_V2


LETTER L002

ORIGINAL LETTER
Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.

V1 (naive prompt, temperature 0.7)
   [rate limited — sleeping 1.5s]
   [rate limited — sleeping 2.2s]
   [rate limited — sleeping 4.5s]
   [rate limited — sleeping 8.2s]
   [rate limited — sleeping 16.1s]
   [rate limited — sleeping 32.9s]


RuntimeError: ask_llm failed after 6 attempts

## Student Reasoning

1. V1 produced three failures that V2 corrected. For L006, V1 states "He has no prior experience." The letter contains no such claim; Kofi writes only "I have not started any of these yet," which refers to the three specific ventures, not to his business history as a whole. V1 turned an inference into a reported fact, and an adverse one. V2 makes no such claim.
V1 never identifies missing information. Neither L002 nor L006 states any income, profit, savings figure or repayment amount, yet V1's summaries pass over these gaps in silence. V2 states them explicitly, "His current financial position, including income, profit, or savings, is not stated" because for a triage tool the absence of information is itself decision-relevant. ."

2. This failure mode is called hallucination, where the model produces content not entailed by the source document.

In [ ]:
import pandas as pd

EXPECTED_KEYS = ["applicant_name", "amount_ghs", "purpose",
                 "monthly_profit_ghs", "has_collateral_or_guarantor",
                 "repayment_months"]

FEWSHOT_LETTER = """Dear Madam,
My name is Ama Serwaa and I bake bread at Tema Community 5. I would like GHS 6,000 to buy a
second oven so I can supply two more shops. My bakery makes about GHS 1,200 profit a month.
My husband, a customs officer, will stand as guarantor. I can repay over 12 months."""

FEWSHOT_JSON = """{
  "applicant_name": "Ama Serwaa",
  "amount_ghs": 6000,
  "purpose": "buy a second oven to supply more shops",
  "monthly_profit_ghs": 1200,
  "has_collateral_or_guarantor": true,
  "repayment_months": 12
}"""

EXTRACT_SYSTEM = """You are a strict data-extraction engine for a microfinance loan system.
You do not converse. You return machine-readable JSON and nothing else.

Output rules:
- Return ONE JSON object only. No markdown fences, no commentary, no trailing text.
- Use EXACTLY these six keys, in this order: applicant_name, amount_ghs, purpose,
  monthly_profit_ghs, has_collateral_or_guarantor, repayment_months.
- If a field is NOT stated in the letter, use null. Do not guess, average, infer or estimate.
- amount_ghs, monthly_profit_ghs and repayment_months must be plain numbers: no currency
  symbols, no thousands separators, no units, no ranges. If the letter gives a range or a
  vague figure ("sometimes around 1,500"), use the single number stated.
- has_collateral_or_guarantor is true ONLY if the letter names specific collateral, a pledged
  asset, a fixed deposit, a group guarantee, or a named guarantor. Optimism, faith,
  trustworthiness and "I am reliable" are NOT collateral, so they give false.
- purpose is a short noun phrase copied from the letter, not a sentence."""

EXTRACT_PROMPT = """Extract the following schema from a loan application letter.

SCHEMA
{{
  "applicant_name": string,
  "amount_ghs": number,
  "purpose": string,
  "monthly_profit_ghs": number or null,
  "has_collateral_or_guarantor": boolean,
  "repayment_months": number or null
}}

EXAMPLE
Letter:
\"\"\"{fewshot_letter}\"\"\"
Output:
{fewshot_json}

NOW EXTRACT FROM THIS LETTER
Letter:
\"\"\"{letter}\"\"\"
Output:"""


def _strip_fences(text):
    """Remove ```json ... ``` fences and any stray prose around the object."""
    t = text.strip()
    t = re.sub(r"^```[a-zA-Z]*\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    start, end = t.find("{"), t.rfind("}")
    return t[start:end + 1] if start != -1 and end != -1 else t


def extract_fields(letter_text, temperature=0.0, label="extract"):
    """Call the LLM and return a dict, or None if the output is not valid JSON."""
    raw_out = ask_llm(
        EXTRACT_PROMPT.format(fewshot_letter=FEWSHOT_LETTER,
                              fewshot_json=FEWSHOT_JSON,
                              letter=letter_text),
        EXTRACT_SYSTEM,
        temperature=temperature, max_tokens=400, label=label,
    )
    try:
        data = json.loads(_strip_fences(raw_out))
    except json.JSONDecodeError as e:
        print(f"WARNING: could not parse JSON ({e}). Raw model output was:\n{raw_out}\n")
        return None
    if not isinstance(data, dict):
        print(f"WARNING: model returned a {type(data).__name__}, not an object.")
        return None
    return {k: data.get(k, None) for k in EXPECTED_KEYS}

extracted = {}
for lid, text in LETTERS.items():
    print(f"Extracting {lid} ")
    extracted[lid] = extract_fields(text, label=f"extract_{lid}")

extract_df = pd.DataFrame.from_dict(extracted, orient="index")[EXPECTED_KEYS]
extract_df.index.name = "letter_id"
display(extract_df)


In [ ]:
BRIEF_SYSTEM = """You are a decision-SUPPORT assistant for a loan officer at a Ghanaian
microfinance institution. Your job is to help a human think, not to think for them.

Hard constraints:
- You NEVER approve, reject, decline, recommend approval, or state a credit decision, a score,
  or a probability of default. The final decision belongs to the human loan officer and to the
  credit committee.
- Every strength and every risk you list must be traceable to words in the letter or to the
  extracted data you are given. If you are relying on something being ABSENT, say so plainly.
- Do not moralise about the applicant, and do not comment on their spelling, grammar or
  writing style. Poor English is not evidence about a business.
- Be concise: at most four bullets per section, one line each."""

BRIEF_PROMPT = """Prepare a decision-support brief for the loan officer.

LETTER
{letter}

EXTRACTED DATA (from the automated extractor; treat it as provisional)
{extracted_json}

Produce EXACTLY these four sections, with these headings:

1. Strengths
   Evidence in the letter that supports the application.

2. Risks
   Concerns an experienced officer should notice, including anything internally inconsistent
   (e.g. proposed repayments that do not match the stated income).

3. Missing information to request
   Specific documents or facts the officer should ask the applicant for.

4. Suggested Next Step
   Choose exactly ONE of: "invite for interview", "request supporting documents",
   "conduct site visit", "flag for senior review", "refer to financial-literacy programme".
   Give one sentence of justification. Do NOT write "approve" or "reject".

End the brief with this line verbatim:
Final Decision: to be made by the human loan officer."""


def make_brief(letter_text, extracted_dict, label="brief"):
    return ask_llm(
        BRIEF_PROMPT.format(
            letter=letter_text,
            extracted_json=json.dumps(extracted_dict, indent=2),
        ),
        BRIEF_SYSTEM,
        temperature=0.0, max_tokens=650, label=label,
    )


briefs = {}
for lid, text in LETTERS.items():
    print(f"Briefing {lid}")
    briefs[lid] = make_brief(text, extracted[lid], label=f"brief_{lid}")

for lid in ["L001", "L002", "L006"]:
    print("\n")
    print(f" DECISION-SUPPORT BRIEF — {lid}")
    print(briefs[lid])

print("\nDECISION-SUPPORT BRIEF — L003 (for the comparison question)")
print(briefs["L003"])

## Student Reasoning Section 3.3

1. The system performed well on L003 and only partially on L006. For L003 every listed strength is grounded in the letter: the GHS 22,000 December revenue, the pledgeable GHS 5,000 fixed deposit, the GHS 2,800 average monthly profit and the 18 months of sales records, and the risk section adds genuine analysis rather than paraphrase, correctly computing that the proposed GHS 1,100 repayment is nearly 40% of stated monthly profit. It omitted one of the accurate strengths in the letter: the business registration number BN-2019-4482. A registered business with a traceable registration is a stronger credit signal than most of what the brief did list, and it also omitted that she employs three apprentices and that the purchase is timed ahead of her demonstrated peak season.

For L006 the risk section is accurate and well-targeted, correctly identifying reliance on friends' opinions rather than evidence, the absence of collateral, and the vagueness of' repaying "when the businesses are booming." The strengths section, however, largely fails. It lists that the applicant is "young and claims to be full of energy," that he "has multiple business ideas," and that he is "aware of the loan officer's advert, indicating some level of initiative." None of these is a strength from a lender's perspective, and the second is arguably a red flag, proposing to split GHS 50,000 across three unstarted ventures indicates unfocused capital allocation, which the brief filed under strengths rather than risks.

2. Practical: the model is reasoning from a single letter. It has no access to credit bureau data, the institution's credit policy and risk appetite, existing portfolio exposure KYC verification, or regulatory lending limits, which are the inputs a real decision actually requires. A recommendation formed without them would look authoritative while resting on a fraction of the evidence.

Ethical: an adverse credit decision materially affects someone's livelihood, and the applicant is entitled to know the basis and to contest it. An automated rejection offers neither; the reasoning is opaque, and the model can assert things the applicant never said, such as "no prior experience," which they cannot rebut because they do not know it was claimed. Accountability also has to rest with a person: a model cannot be held responsible, so a decision it makes is a decision no one is answerable for.

## Commit Hash
b89a77e5651a3ba0071dbb5b98f89c3807708566

In [ ]:
# Part 4.1
STOPWORDS = {"a", "an", "the", "and", "or", "of", "to", "for", "into", "with", "my", "new"}


def _norm_words(s):
    return {w for w in re.findall(r"[a-z]+", str(s).lower())} - STOPWORDS


def field_matches(field, predicted, gold):
    if field == "applicant_name":
        return str(predicted).strip().lower() == str(gold).strip().lower()
    if field == "purpose":
        p, g = _norm_words(predicted), _norm_words(gold)
        if not g:
            return not p
        return len(p & g) / len(p | g) >= 0.30
    if predicted is None or gold is None:
        return predicted is None and gold is None
    if isinstance(gold, bool):
        return bool(predicted) is gold
    try:
        return float(predicted) == float(gold)
    except (TypeError, ValueError):
        return False


gold_ids = list(GOLD.keys())
rows = {}
for field in EXPECTED_KEYS:
    row = {}
    for lid in gold_ids:
        pred = (extracted[lid] or {}).get(field)
        row[lid] = field_matches(field, pred, GOLD[lid][field])
    row["accuracy"] = sum(row[l] for l in gold_ids) / len(gold_ids)
    rows[field] = row

accuracy_df = pd.DataFrame.from_dict(rows, orient="index")
accuracy_df.index.name = "field"
print("Per-field correctness (True = matches gold)\n")
display(accuracy_df)

overall = accuracy_df["accuracy"].mean()
print(f"\nOverall extraction accuracy across {len(EXPECTED_KEYS)} fields "
      f"x {len(gold_ids)} letters = {overall:.1%}")

print("\nMismatches (field | letter | predicted | gold):")
found = False
for field in EXPECTED_KEYS:
    for lid in gold_ids:
        if not rows[field][lid]:
            found = True
            print(f"  {field:<28} | {lid} | {(extracted[lid] or {}).get(field)!r:<45} "
                  f"| {GOLD[lid][field]!r}")
if not found:
    print("  none. every field matched.")

In [ ]:
# Part 4.2 — Reliability
N = 5
reliability = {}

for temp in (0.0, 1.0):
    print(f"Running extract_fields on L004, {N} times at temperature={temp}")
    outs = [extract_fields(LETTERS["L004"], temperature=temp, label=f"rel_{temp}")
            for _ in range(N)]
    reliability[temp] = outs

for temp, outs in reliability.items():
    valid = [o for o in outs if o is not None]
    fingerprints = [json.dumps(o, sort_keys=True) for o in valid]
    unique = set(fingerprints)
    print(f"\nTEMPERATURE = {temp}")
    print(f"  valid JSON            : {len(valid)}/{N}")
    print(f"  distinct results      : {len(unique)}")
    print(f"  identical across runs : {'YES' if len(unique) == 1 else 'NO'}")
    for i, fp in enumerate(sorted(unique), 1):
        print(f"    variant {i} (x{fingerprints.count(fp)}): {fp}")

# Which fields actually moved?
print("\nField-by-field variability:")
for temp, outs in reliability.items():
    valid = [o for o in outs if o]
    print(f"  temperature {temp}:")
    for k in EXPECTED_KEYS:
        vals = {json.dumps(o.get(k)) for o in valid}
        flag = "" if len(vals) <= 1 else " VARIES"
        print(f"    {k:<28} {len(vals)} distinct value(s){flag}")

In [ ]:
# TEST 1 — ask the summarizer for a detail that does not exist in L001.
#          A trustworthy system says "not stated"; a hallucinating one

print("TEST 1 — asking for a fact that is NOT in letter L001 (credit score)")


probe_1 = ask_llm(
    "What is the applicant's credit score, and how many years of bank statements did "
    "they attach?\n\nLOAN APPLICATION\n" + LETTERS["L001"],
    SUMMARY_SYSTEM,
    temperature=0.0, max_tokens=200, label="probe_credit_score",
)
print(probe_1)

refusal_markers = ["not stated", "not mention", "does not state", "no mention",
                   "not provided", "not included", "not specified", "not available",
                   "not indicated", "no credit score", "not attach"]
test1_pass = any(m in probe_1.lower() for m in refusal_markers) and \
             not re.search(r"credit score (of|is) \s*\d", probe_1.lower())
print(f"\nRESULT: {'PASS' if test1_pass else 'FAIL'} "
      f"— {'admitted the information is absent' if test1_pass else 'invented a detail'}")


# TEST 2 — feed the extractor text that is not a loan application at all.
#          A trustworthy system returns nulls; a hallucinating one
#          fabricates an applicant.
print("TEST 2 — feeding the extractor an irrelevant document (a weather report)")

IRRELEVANT = """Ghana Meteorological Agency — Regional Outlook
Accra will be partly cloudy today with a high of 31 degrees Celsius and a 40 percent chance
of afternoon showers. Winds from the south-west at 15 km/h. Coastal waters moderate.
Tomorrow: scattered thunderstorms over the Volta Region, easing by Thursday."""

probe_2 = extract_fields(IRRELEVANT, label="probe_irrelevant")
print("Extractor returned:")
print(json.dumps(probe_2, indent=2) if probe_2 else "None (unparseable output)")

if probe_2 is None:
    test2_pass = True
    verdict = "refused to produce an object at all (acceptable: nothing fabricated)"
else:
    fabricated = [k for k, v in probe_2.items() if v not in (None, "", False)]
    test2_pass = len(fabricated) == 0
    verdict = ("returned all nulls / false — nothing fabricated" if test2_pass
               else f"fabricated values for: {fabricated}")
print(f"\nRESULT: {'PASS' if test2_pass else 'FAIL'} — {verdict}")

print("SUMMARY OF ADVERSARIAL TESTS")
print(f"  Test 1 (missing detail in L001) : {'PASS' if test1_pass else 'FAIL'}")
print(f"  Test 2 (irrelevant input)       : {'PASS' if test2_pass else 'FAIL'}")
print("\nThe verbatim outputs above are the evidence for the reasoning box below.")

In [ ]:

# Section 5, Q3 — Cost and scale, computed from the real usage numbers
usage_df = pd.DataFrame(TOKEN_LOG,
                        columns=["label", "prompt_tokens",
                                 "completion_tokens", "total_tokens"])
usage_df["stage"] = usage_df["label"].str.replace(r"_L\d+$", "", regex=True)

print(f"Total API calls this notebook : {len(usage_df)}")
print(f"Total tokens consumed         : {usage_df['total_tokens'].sum():,}")
display(usage_df.groupby("stage")[["prompt_tokens", "completion_tokens", "total_tokens"]]
        .agg(["count", "sum", "mean"]).round(1))

stage_mean = usage_df.groupby("stage")["total_tokens"].mean()
tokens_per_app = sum(stage_mean.get(s, 0) for s in ["summary_v2", "extract", "brief"])
print(f"\nTokens to process ONE application (summary + extraction + brief): "
      f"{tokens_per_app:,.0f}")
print(f"Tokens for 1,000 applications per month: {tokens_per_app * 1000:,.0f}")

# Rough price comparison
PRICES = {"Groq llama-3.3-70b": 0.79, "GPT-4o-class": 5.00, "Small/8B model": 0.10}
print("\nIndicative monthly cost at 1,000 applications:")
for name, usd_per_m in PRICES.items():
    print(f"  {name:<22} ~ US${tokens_per_app * 1000 / 1_000_000 * usd_per_m:,.2f}")